In [83]:
# !pip install googletrans==4.0.0-rc1

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from scipy.signal import savgol_filter
import numpy as np
from scipy.signal import hilbert
# from googletrans import Translator

import plotly.graph_objects as go
import plotly.subplots as sp
import numpy as np
import itertools

#Read data

In [84]:
#Read data
# File paths
mof_file_path = "/content/MOF"
output_path = "/content/MOF_extracted_final.xlsx"

# Load the health indicator data
hi_file_path = '/content/HIDataWithTrajectory.xlsx'
health_data = pd.read_excel(hi_file_path)

#Extract the MOF information

In [ ]:
def extract_and_separate_mof_data(file_path: str, output_path: str, trajectory_data: dict, data: pd.DataFrame) -> dict:
    """
    Extract MOF (Modes of Failure), startingTime, endingTime, and Time from a raw file, clean the data,
    and split into separate lists based on trajectory values.

    Args:
        file_path (str): Path to the input raw file containing data.
        output_path (str): Path where the cleaned Excel file will be saved.
        trajectory_data (dict): A dictionary containing trajectory columns (Cr1, Cr2, Cv2, Sys).
        data (pd.DataFrame): DataFrame containing the health indicator, time, and other columns.

    Returns:
        dict: A dictionary containing separated MOF, startingTime, endingTime, and Time lists for Cr1, Cr2, Cv2, and Sys.
    """
    # Read raw file
    with open(file_path, 'r', encoding='utf-8') as f:
        raw_data = f.readlines()

    # Convert raw data to a DataFrame
    raw_df = pd.DataFrame(raw_data, columns=['raw_text'])

    # Filter rows containing meaningful data based on the '|' delimiter
    data_filtered = raw_df[raw_df['raw_text'].str.contains(r'\|', na=False)]

    # Split rows into columns based on the '|' delimiter
    split_data = data_filtered['raw_text'].str.split('|', expand=True)

    # Strip whitespace and clean columns
    split_data = split_data.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    # Assign dynamic column names
    split_data.columns = [f"Column_{i}" for i in range(split_data.shape[1])]

    # Extract relevant columns for MOF, startingTime, and endingTime
    relevant_data = split_data[['Column_2', 'Column_3', 'Column_4']]
    relevant_data.columns = ['MOF', 'startingTime', 'endingTime']

    # Clean the data by removing headers and invalid rows
    relevant_data_cleaned = relevant_data[~relevant_data['MOF'].str.contains('MOF|No', na=False)]

    # Add prefixes/suffixes
    mof_list = ['MOF starts'] + relevant_data_cleaned['MOF'].tolist()
    starting_time_list = ['Beginning'] + relevant_data_cleaned['startingTime'].tolist()
    ending_time_list = relevant_data_cleaned['endingTime'].tolist() + ['Ending']

    # Save the cleaned data to an Excel file
    relevant_data_cleaned.to_excel(output_path, index=False)

    # Function to separate sequences
    def separate_sequences(trajectory, original_list, separated_list):
        unique_values = trajectory.unique()
        for value in unique_values:
            sequence = pd.Series(original_list)[trajectory == value].reset_index(drop=True)
            separated_list.append(sequence.tolist())

    # Initialize separated lists
    separated_mof = {"Cr1": [], "Cr2": [], "Cv2": [], "Sys": []}
    separated_starting_time = {"Cr1": [], "Cr2": [], "Cv2": [], "Sys": []}
    separated_ending_time = {"Cr1": [], "Cr2": [], "Cv2": [], "Sys": []}
    separated_time = {"Cr1": [], "Cr2": [], "Cv2": [], "Sys": []}

    # Separate MOF, startingTime, endingTime, and Time based on trajectory values
    separate_sequences(trajectory_data["Cr1"], mof_list, separated_mof["Cr1"])
    separate_sequences(trajectory_data["Cr2"], mof_list, separated_mof["Cr2"])
    separate_sequences(trajectory_data["Cv2"], mof_list, separated_mof["Cv2"])
    # separate_sequences(trajectory_data["Sys"], mof_list, separated_mof["Sys"])

    separate_sequences(trajectory_data["Cr1"], starting_time_list, separated_starting_time["Cr1"])
    separate_sequences(trajectory_data["Cr2"], starting_time_list, separated_starting_time["Cr2"])
    separate_sequences(trajectory_data["Cv2"], starting_time_list, separated_starting_time["Cv2"])
    # separate_sequences(trajectory_data["Sys"], starting_time_list, separated_starting_time["Sys"])

    separate_sequences(trajectory_data["Cr1"], ending_time_list, separated_ending_time["Cr1"])
    separate_sequences(trajectory_data["Cr2"], ending_time_list, separated_ending_time["Cr2"])
    separate_sequences(trajectory_data["Cv2"], ending_time_list, separated_ending_time["Cv2"])
    # separate_sequences(trajectory_data["Sys"], ending_time_list, separated_ending_time["Sys"])

    separate_sequences(trajectory_data["Cr1"], data["Time"], separated_time["Cr1"])
    separate_sequences(trajectory_data["Cr2"], data["Time"], separated_time["Cr2"])
    separate_sequences(trajectory_data["Cv2"], data["Time"], separated_time["Cv2"])
    # separate_sequences(trajectory_data["Sys"], data["Time"], separated_time["Sys"])

    # Create health indicator sequences
    Cr1_sequences, Cr2_sequences, Cv2_sequences, Sys_sequences = [], [], [], []
    separate_sequences(trajectory_data["Cr1"], data["OriginalHICr1"], Cr1_sequences)
    separate_sequences(trajectory_data["Cr2"], data["OriginalHICr2"], Cr2_sequences)
    separate_sequences(trajectory_data["Cv2"], data["OriginalHICv2"], Cv2_sequences)
    # separate_sequences(trajectory_data["Sys"], data["System_HI"], Sys_sequences)

    # Return the results
    return {
        "MOF": separated_mof,
        "startingTime": separated_starting_time,
        "endingTime": separated_ending_time,
        "Time": separated_time,
        "HI": {
            "Cr1": Cr1_sequences,
            "Cr2": Cr2_sequences,
            "Cv2": Cv2_sequences,
            # "Sys": Sys_sequences
        }
    }

trajectory_data = {
    "Cr1": health_data['trajectoryCr1'],
    "Cr2": health_data['trajectoryCr2'],
    "Cv2": health_data['trajectoryCv2'],
    # "Sys": health_data['trajectorySys'],
    "Time": health_data['Time']
}


# Extract and separate MOF data
result = extract_and_separate_mof_data(mof_file_path, output_path, trajectory_data, health_data)
result_keys = list(result.keys())
sub_result_keys = {key: list(result[key].keys()) if isinstance(result[key], dict) else None for key in result_keys}
print("result_keys:", result_keys)
print("sub_result_keys:", sub_result_keys)
# Outputs example
print("MOF for Cr1:", result["MOF"]["Cr1"][:5])  # First 5 MOF for Cr1
print("Starting Times for Cr2:", result["startingTime"]["Cr2"][:5])
print("Ending Times for Cv2:", result["endingTime"]["Cv2"][:5])
print("Time for Sys:", result["Time"]["Sys"][:5])
# print("Health Indicators for Sys:", result["HI"]["Sys"][:5])


<ipython-input-3-51e5552041e1>:29: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  split_data = split_data.applymap(lambda x: x.strip() if isinstance(x, str) else x)


result_keys: ['MOF', 'startingTime', 'endingTime', 'Time', 'HI']
sub_result_keys: {'MOF': ['Cr1', 'Cr2', 'Cv2', 'Sys'], 'startingTime': ['Cr1', 'Cr2', 'Cv2', 'Sys'], 'endingTime': ['Cr1', 'Cr2', 'Cv2', 'Sys'], 'Time': ['Cr1', 'Cr2', 'Cv2', 'Sys'], 'HI': ['Cr1', 'Cr2', 'Cv2']}
MOF for Cr1: [['MOF starts', 'Válvula defectuosa', 'Mantención programada', 'Mantención programada', 'Mantención programada', 'Mantención programada', 'Mantención programada', 'Mantención programada', 'Mantención programada', 'Mantención programada', 'Falta de aceite hidráulico', 'Mantención programada', 'Mantención programada', 'Mantención programada', 'Mantención programada', 'Lubricación  deficiente', 'Temperatura fuera de rango', 'Bomba hidráulica con fuga', 'Fuga de aceite', 'Falla en sistema hidráulico', 'Fuga de aceite', 'Falla sistema de hidroset-poste', 'Falla sistema de hidroset-poste', 'Falla sistema de hidroset-poste', 'Mantención programada', 'Válvula defectuosa', 'Válvula defectuosa', 'Mantención pro

#Prepare all the data

In [ ]:
# Dynamically create variable names and assign corresponding values from `result`
variables = {}
for key, sub_keys in sub_result_keys.items():
    for sub_key in sub_keys:
        variable_name = f"{key}_{sub_key}"
        variables[variable_name] = result[key][sub_key]

# Extract variables for further use
MOF_Cr1 = variables['MOF_Cr1']
MOF_Cr2 = variables['MOF_Cr2']
MOF_Cv2 = variables['MOF_Cv2']
# MOF_Sys = variables['MOF_Sys']

startingTime_Cr1 = variables['startingTime_Cr1']
startingTime_Cr2 = variables['startingTime_Cr2']
startingTime_Cv2 = variables['startingTime_Cv2']
# startingTime_Sys = variables['startingTime_Sys']

endingTime_Cr1 = variables['endingTime_Cr1']
endingTime_Cr2 = variables['endingTime_Cr2']
endingTime_Cv2 = variables['endingTime_Cv2']
# endingTime_Sys = variables['endingTime_Sys']

Time_Cr1 = variables['Time_Cr1']
Time_Cr2 = variables['Time_Cr2']
Time_Cv2 = variables['Time_Cv2']
Time_Sys = variables['Time_Sys']

HI_Cr1 = variables['HI_Cr1']
HI_Cr2 = variables['HI_Cr2']
HI_Cv2 = variables['HI_Cv2']
# HI_Sys = variables['HI_Sys']


In [231]:
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

def moving_average(data, window_size):
    """
    Compute the moving average of a 1D array.

    Args:
        data (array): The input array.
        window_size (int): The size of the moving window.

    Returns:
        array: Smoothed array with the same length as the input.
    """
    if len(data) < window_size:
        return data  # Return the original array if it's too short
    cumsum = np.cumsum(np.insert(data, 0, 0))
    moving_avg = (cumsum[window_size:] - cumsum[:-window_size]) / window_size
    pad_width = len(data) - len(moving_avg)
    return np.pad(moving_avg, (pad_width // 2, pad_width - pad_width // 2), mode='edge')

def ema_smooth(data, alpha=0.1):
    """
    Smooth a 1D array using Exponential Moving Average (EMA).

    Args:
        data (array): The input array.
        alpha (float): Smoothing factor, where 0 < alpha <= 1. Smaller values give smoother results.

    Returns:
        array: Smoothed array.
    """
    smoothed = np.copy(data)
    for i in range(1, len(data)):
        smoothed[i] = alpha * data[i] + (1 - alpha) * smoothed[i - 1]
    return smoothed

def normalize_and_smooth(data):
    """
    Normalize and smooth a list of lists using EMA while retaining trends and valleys' amplitudes.
    Also compute the lower envelope spectrum of the processed data.

    Args:
        data (list): A list of lists containing sequences to be normalized and smoothed.

    Returns:
        tuple: Two lists of lists: smoothed data and lower envelope spectra.
    """
    smoothed_data = []
    lower_envelopes = []
    ema_alpha = 0.05  # Smoothing factor for EMA

    for sequence in data:
        if len(sequence) == 0 or np.all(np.isnan(sequence)):
            smoothed_data.append(sequence)
            lower_envelopes.append(sequence)
            continue

        flat_sequence = pd.Series(sequence).ffill().bfill().values
        nan_mask = np.isnan(sequence)
        non_nan_values = flat_sequence[~nan_mask]

        if len(non_nan_values) == 0:
            smoothed_data.append(sequence)
            lower_envelopes.append(sequence)
            continue

        min_val = np.min(non_nan_values)
        max_val = np.max(non_nan_values)
        normalized = (non_nan_values - min_val) / (max_val - min_val)

        if len(normalized) <= 2:
            smoothed = normalized
        else:
            smoothed = ema_smooth(normalized, alpha=ema_alpha)

        lower_envelope = moving_average(smoothed, window_size=5)

        # Ensure length consistency
        if len(lower_envelope) < len(non_nan_values):
            lower_envelope = np.pad(lower_envelope, (0, len(non_nan_values) - len(lower_envelope)), mode='edge')
        elif len(lower_envelope) > len(non_nan_values):
            lower_envelope = lower_envelope[:len(non_nan_values)]

        restored_sequence = np.full_like(sequence, np.nan, dtype=float)
        restored_sequence[~nan_mask] = smoothed

        restored_lower_envelope = np.full_like(sequence, np.nan, dtype=float)
        restored_lower_envelope[~nan_mask] = lower_envelope

        smoothed_data.append(restored_sequence.tolist())
        lower_envelopes.append(restored_lower_envelope.tolist())

    return smoothed_data, lower_envelopes

# Process the data and generate new lists with prefixed names
Nor_smooth_HI_Cr1, lower_envelopes_Nor_smooth_HI_Cr1 = normalize_and_smooth(HI_Cr1)
Nor_smooth_HI_Cr2, lower_envelopes_Nor_smooth_HI_Cr2 = normalize_and_smooth(HI_Cr2)
Nor_smooth_HI_Cv2, lower_envelopes_Nor_smooth_HI_Cv2 = normalize_and_smooth(HI_Cv2)
# Nor_smooth_HI_Sys, lower_envelopes_Nor_smooth_HI_Sys = normalize_and_smooth(HI_Sys)

print(f"Nor_smooth_HI_Cr1: {Nor_smooth_HI_Cr1[:1]}")
print(f"Lower_envelopes_Nor_smooth_HI_Cr1: {lower_envelopes_Nor_smooth_HI_Cr1[:1]}")
print(f"Nor_smooth_HI_Cr2: {Nor_smooth_HI_Cr2[:1]}")
print(f"Lower_envelopes_Nor_smooth_HI_Cr2: {lower_envelopes_Nor_smooth_HI_Cr2[:1]}")


Nor_smooth_HI_Cr1: [[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, 0.6358631696128432, 0.6357666935041452, 0.6356976232938586, 0.6313454975279799, 0.6270093474510005, 0.624397457471384, 0.6212228970944145, 0.6183747690279445, 0.6157846681754262, 0.6113251734187636, 0.6060586280446827, 0.6008873241961663, 0.5976349276407731, 0.5951053904246225, 0.594877587784582, 0.5950642402885056, 0.5951312487656526, 0.595991511065665, 0.5973837310847846, 0.5988034369203892, 0.6005577564186398, 0.6015430525503244, 0.6031932459661024, 0.6032537186416964, 0.6026443698739186, 0.6023990209882585, 0.6033071657389505, 0.6051805201248213, 0.6077105974999376, 0.6110470632742753, 0.614477123165011, 0.6181905235578673, 0.6213572454397851, 0.625495

#Plot

In [233]:


def plot_flexible_subplots(
    Time_Cr1, Time_Cr2, Time_Cv2,
    lower_envelopes_Nor_smooth_HI_Cr1,
    lower_envelopes_Nor_smooth_HI_Cr2,
    lower_envelopes_Nor_smooth_HI_Cv2,
    startingTime_Cr1, startingTime_Cr2, startingTime_Cv2,
    MOF_Cr1, MOF_Cr2, MOF_Cv2,
    segment=slice(0, 3)
):
    """
    This function creates a set of subplots displaying Health Indicator (HI) data for different conditions (Cr1, Cr2, Cv2)
    and plots the system behavior in a fourth subplot. It allows visualizing data in an organized and flexible way,
    including annotations for specific events.

    Parameters:
    - Time_Cr1, Time_Cr2, Time_Cv2: Time data for each condition (Cr1, Cr2, Cv2)
    - lower_envelopes_Nor_smooth_HI_Cr1, lower_envelopes_Nor_smooth_HI_Cr2, lower_envelopes_Nor_smooth_HI_Cv2: Health Indicator data
    - startingTime_Cr1, startingTime_Cr2, startingTime_Cv2: Starting times for each condition
    - MOF_Cr1, MOF_Cr2, MOF_Cv2: Multi-Objective Function data for each condition
    - segment: Defines the portion of the data to plot, default is slice(0, 3)
    """
    # Initialize subplots with shared x-axis
    fig = sp.make_subplots(
        rows=4, cols=1, shared_xaxes=True,
    )

    colors = ['blue', 'orange', 'green', 'red']
    markers = ['circle', 'square', 'triangle-up', 'diamond']
    y_titles = ["HI: Cr1", "HI: Cr2", "HI: Cv2", "HI: Sys"]

    # Data segments for the first three subplots
    data_lists = [
        (Time_Cr1[segment], lower_envelopes_Nor_smooth_HI_Cr1[segment], MOF_Cr1[segment], y_titles[0], colors[0], markers[0]),
        (Time_Cr2[segment], lower_envelopes_Nor_smooth_HI_Cr2[segment], MOF_Cr2[segment], y_titles[1], colors[1], markers[1]),
        (Time_Cv2[segment], lower_envelopes_Nor_smooth_HI_Cv2[segment], MOF_Cv2[segment], y_titles[2], colors[2], markers[2])
    ]

    # Plot each data segment
    for i, (time_data, hi_data, MOF, y_title, color, marker) in enumerate(data_lists, start=1):
        for t_seq, hi_seq, mof_seq in zip(time_data, hi_data, MOF):
            fig.add_trace(go.Scatter(
                x=np.array(t_seq), y=hi_seq,
                mode='lines+markers',
                line=dict(color=color),
                marker=dict(symbol=marker, size=2),
                name=None,  # No legend
                showlegend=False
            ), row=i, col=1)
            first_word, *remaining_words = str(mof_seq).split()

            # Generate a shortened version for the annotation
            abbreviation = "".join([word[0].upper() for word in (first_word, *remaining_words)])  # Abbreviate by initials

            annotation_position = len(t_seq) // 2
            annotation_y = 1.15
            fig.add_annotation(
                x=t_seq[annotation_position], y=annotation_y,  # Place slightly above the data point
                text=f"{abbreviation}",
                showarrow=False,
                font=dict(size=14, color="black"),
                row=i, col=1
            )

    # Flatten lists for the fourth subplot
    Time_Cr1_flat = list(itertools.chain.from_iterable(Time_Cr1[segment]))
    Time_Cr2_flat = list(itertools.chain.from_iterable(Time_Cr2[segment]))
    Time_Cv2_flat = list(itertools.chain.from_iterable(Time_Cv2[segment]))

    HI_Cr1_flat = list(itertools.chain.from_iterable(lower_envelopes_Nor_smooth_HI_Cr1[segment]))
    HI_Cr2_flat = list(itertools.chain.from_iterable(lower_envelopes_Nor_smooth_HI_Cr2[segment]))
    HI_Cv2_flat = list(itertools.chain.from_iterable(lower_envelopes_Nor_smooth_HI_Cv2[segment]))

    startingTime_Cr1_flat = list(itertools.chain.from_iterable(startingTime_Cr1[segment]))
    startingTime_Cr2_flat = list(itertools.chain.from_iterable(startingTime_Cr2[segment]))
    startingTime_Cv2_flat = list(itertools.chain.from_iterable(startingTime_Cv2[segment]))

    longest_time = max(len(Time_Cr1_flat), len(Time_Cr2_flat), len(Time_Cv2_flat))
    min_values_based_Sys_HI = []
    min_values_base_Sys_Time = []
    min_values_base_Sys_startingTime = []

    for i in range(longest_time):
        v1 = HI_Cr1_flat[i] if i < len(HI_Cr1_flat) else np.nan
        v2 = HI_Cr2_flat[i] if i < len(HI_Cr2_flat) else np.nan
        v3 = HI_Cv2_flat[i] if i < len(HI_Cv2_flat) else np.nan

        t1 = Time_Cr1_flat[i] if i < len(Time_Cr1_flat) else np.nan
        t2 = Time_Cr2_flat[i] if i < len(Time_Cr2_flat) else np.nan
        t3 = Time_Cv2_flat[i] if i < len(Time_Cv2_flat) else np.nan

        s1 = startingTime_Cr1[i] if i < len(startingTime_Cr1_flat) else None
        s2 = startingTime_Cr2[i] if i < len(startingTime_Cr2_flat) else None
        s3 = startingTime_Cv2[i] if i < len(startingTime_Cv2_flat) else None

        available_values = [v for v in [v1, v2, v3] if not np.isnan(v)]

        if available_values:
            min_value = min(available_values)
            min_values_based_Sys_HI.append(min_value)

            if min_value == v1:
                min_values_base_Sys_Time.append(t1)
                min_values_base_Sys_startingTime.append(s1)
            elif min_value == v2:
                min_values_base_Sys_Time.append(t2)
                min_values_base_Sys_startingTime.append(s2)
            elif min_value == v3:
                min_values_base_Sys_Time.append(t3)
                min_values_base_Sys_startingTime.append(s3)
        else:
            min_values_based_Sys_HI.append(np.nan)
            fallback_time = t1 if not np.isnan(t1) else (t2 if not np.isnan(t2) else t3)
            min_values_base_Sys_Time.append(fallback_time)
            min_values_base_Sys_startingTime.append(None)

    # Plot the fourth subplot
    fig.add_trace(go.Scatter(
        x=min_values_base_Sys_Time, y=min_values_based_Sys_HI,
        mode='markers',
        line=dict(color=colors[3]),
        marker=dict(symbol=markers[3], size=2),
        name=None,  # No legend
        showlegend=False
    ), row=4, col=1)

    unique_times = list(set(filter(None, min_values_base_Sys_startingTime)))  # Remove None values
    unique_times.sort()  # Sort time points
    print(unique_times)

    if unique_times:
        # Get valid time point for labeling
        valid_times = list(filter(lambda x: x is not None and isinstance(x, (int, float)), min_values_base_Sys_Time))

        if valid_times:
            start_time = valid_times[0]
            mid_time = valid_times[len(valid_times) // 2]
            end_time = valid_times[-1]

            begin = unique_times[0]
            middle = unique_times[len(unique_times) // 2]
            end = unique_times[-1]  # 结尾

            # Find the closest match for the beginning, middle and end time points in min_values_base_Sys_Time
            for target_time, labels in zip([start_time, mid_time, end_time], [begin, middle, end]):
                annotation_x = target_time

                # 添加标注
                fig.add_annotation(
                    x=annotation_x, y=0,  # Place annotation at y=0
                    text=f"{labels}",
                    showarrow=False,
                    font=dict(size=14, color="black"),
                    xanchor="center",
                    yanchor="top",
                    row=4, col=1
                )

    # Update layout
    fig.update_layout(
        height=800, width=1000,
        title=None,  # No central title
        plot_bgcolor="white",
        showlegend=False  # Disable legend globally
    )

    # Add y-axis titles for each subplot
    for i, y_title in enumerate(y_titles, start=1):
        fig['layout'][f'yaxis{i}'].update(title=y_title, range=[0, 1.2])

    # Hide x-axis for the first 3 subplots and set only title for the 4th subplot
    for i in range(1, 4):  # Subplots 1 to 3
        fig['layout'][f'xaxis{i}'].update(showticklabels=False, title=None)
    fig['layout']['xaxis4'].update(title="Time", showticklabels=False,title_standoff=25)  # Enable tick labels for 4th

    fig.show()



In [234]:
plot_flexible_subplots(
    Time_Cr1, Time_Cr2, Time_Cv2,
    lower_envelopes_Nor_smooth_HI_Cr1,
    lower_envelopes_Nor_smooth_HI_Cr2,
    lower_envelopes_Nor_smooth_HI_Cv2,
    startingTime_Cr1[0], startingTime_Cr2[0], startingTime_Cv2[0],
    MOF_Cr1[0], MOF_Cr2[0], MOF_Cv2[0],
    segment=slice(0, 5)# Adjust the slice here to choose showing which curves,
)
#MS:Start to apply maintenance plan
#VD:Válvula defectuosa
#MP:Mantención programada

['2023-06-01 05:27:00', '2023-06-08 05:00:00', '2023-06-15 04:40:00', '2023-06-29 04:55:00', '2023-07-06 04:47:00', '2023-07-11 19:01:00', '2023-07-17 20:00:00', '2023-08-19 14:01:00', '2023-08-21 17:42:00', '2023-08-21 20:00:00', '2023-08-22 04:45:00', '2023-08-23 03:51:00', '2023-08-25 22:44:00', '2023-08-30 04:40:00', '2023-08-30 05:47:00', '2023-09-06 06:19:00', '2023-09-13 08:00:00', '2023-09-18 14:27:00', '2023-09-20 08:00:00', '2023-09-27 03:35:00', '2023-10-12 08:00:00', '2023-10-14 00:06:00']


In [ ]:
# def polynomial_trend(data, degree=5):
#     """
#     Extract the trend using polynomial fitting.

#     Args:
#         data (array): The input array.
#         degree (int): Degree of the polynomial.

#     Returns:
#         array: The extracted trend.
#     """
#     x = np.arange(len(data))
#     coeffs = np.polyfit(x, data, degree)
#     poly = np.poly1d(coeffs)
#     return poly(x)
# def fast_trend_polynomial(data, degree=3):
#     """
#     Quickly extract trends for a list of sequences using polynomial fitting.

#     Args:
#         data (list): A list of lists containing sequences.
#         degree (int): Degree of the polynomial.

#     Returns:
#         list: Extracted trends for all sequences.
#     """
#     trends = []
#     for sequence in data:
#         # Handle NaN values and flatten the sequence
#         nan_mask = np.isnan(sequence)  # Track original NaNs
#         flat_sequence = pd.Series(sequence).ffill().bfill().values  # Handle NaNs

#         if len(flat_sequence) == 0:
#             # If the sequence is empty, append as is
#             trends.append(sequence)
#             continue

#         # Compute polynomial trend
#         trend = polynomial_trend(flat_sequence, degree=degree)

#         # Restore original structure with NaNs
#         restored_trend = np.full_like(sequence, np.nan, dtype=float)
#         restored_trend[~nan_mask] = trend[:np.sum(~nan_mask)]  # Match valid entries
#         trends.append(restored_trend.tolist())
#     return trends



# trend_HI_Cr1 = fast_trend_polynomial(HI_Cr1, degree=3)
# trend_HI_Cr2 = fast_trend_polynomial(HI_Cr2, degree=3)
# trend_HI_Cv2 = fast_trend_polynomial(HI_Cv2, degree=3)
# trend_HI_Sys = fast_trend_polynomial(HI_Sys, degree=3)
